# Lab 06 — Decision Trees and k-Nearest Neighbours

**Class 06 · Topic 5.** Guide: `practice/06_trees_knn.pdf`. Lecture: `slides/06_trees_knn.pdf`. Demos: `01_decision_trees.ipynb`
and `02_knn.ipynb`.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready: it draws
the plots and prints the comparisons once your functions work.

**The rule of this lab.** Trees and KNN have no gradient to descend: they are *algorithms*. You write them with NumPy
(the tree) and `jax.numpy` (the distances of KNN), and compare each with its scikit-learn twin on the **same split and the
same scaling**. The notebook prints the difference; explain it.

In [ ]:
import time
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from lab06_utils import (
    compare,
    fit_standardizer,
    leaves_of,
    load_breast_cancer,
    load_concrete,
    load_digits,
    load_german_credit,
    plot_curves,
    plot_pred_vs_actual,
    print_tree,
    standardize,
    stratified_split,
)
from scipy import stats
from sklearn.datasets import make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(40)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
_ = pl.Config.set_tbl_hide_dataframe_shape(True)

# Part A — Data visualization and evaluation methodology

Three data sets: **breast cancer** (30 measurements, benign or malignant), **German credit** (1000 applications,
61 columns after encoding, 30% bad credits) and **concrete** (1030 mixtures, the strength in MPa). The split is
stratified; the standardization statistics (needed for KNN, harmless for trees) come from the **training** part only.

In [ ]:
Xb, yb, names_b = load_breast_cancer()
Xg, yg, names_g = load_german_credit()
Xc, yc, names_c = load_concrete()
Xd, yd, _ = load_digits()
print(f"breast cancer {Xb.shape}, benign {yb.mean():.1%}; german credit {Xg.shape}, bad {yg.mean():.1%}; concrete {Xc.shape}")

rng = np.random.default_rng(0)
trb, teb = stratified_split(yb, 0.25, rng)
trg, teg = stratified_split(yg, 0.25, rng)
trc, tec = np.random.default_rng(1).permutation(len(yc))[: int(0.75 * len(yc))], np.random.default_rng(1).permutation(len(yc))[int(0.75 * len(yc)):]
print(f"baselines (always the majority class): breast cancer {max(yb[teb].mean(), 1 - yb[teb].mean()):.3f}, "
      f"german credit {max(yg[teg].mean(), 1 - yg[teg].mean()):.3f}; concrete: predict the mean, RMSE {np.sqrt(np.mean((yc[trc].mean() - yc[tec]) ** 2)):.2f} MPa")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
j = names_b.index("worst_perimeter")
axes[0].hist([Xb[yb == 1, j], Xb[yb == 0, j]], bins=25, stacked=True, label=["benign", "malignant"])
axes[0].set(xlabel="worst_perimeter", title="breast cancer: one feature, two classes")
axes[0].legend()
j2 = names_g.index("duration")
axes[1].hist([Xg[yg == 0, j2], Xg[yg == 1, j2]], bins=20, stacked=True, label=["good", "bad"])
axes[1].set(xlabel="duration (months)", title="german credit")
axes[1].legend()
axes[2].scatter(Xc[:, names_c.index("age")], yc, s=6)
axes[2].set(xlabel="age (days)", ylabel="strength (MPa)", title="concrete: a non-linear relation")
plt.tight_layout()
plt.show()

**Answers (A).**

1. Always answering the majority class gives 0.627 on breast cancer and **0.700** on German credit: any German-credit model must beat 0.70 before it says anything, and the same for the concrete data, where predicting the mean costs an RMSE of 15.8 MPa.
2. The benign and malignant histograms of `worst_perimeter` overlap only in a narrow band: a **single threshold** (a single question of a tree) separates most of the classes. The concrete scatter against `age` is curved and noisy: a line would fit it poorly, a tree (a staircase) fits any shape but needs many steps.

# Part B — Impurity and the best split

A node holds the examples that reached it, with class counts $n_1,\dots,n_K$ ($p_k = n_k/n$). Its impurity is the
**Gini index** $G = 1 - \sum_k p_k^2$ or the **entropy** $H = -\sum_k p_k\log_2 p_k$ (with $0\log 0 = 0$). The **gain** of a
question "$x \le t$" is $\ \text{impurity(parent)} - \frac{n_L}{n}\,\text{impurity(left)} - \frac{n_R}{n}\,\text{impurity(right)}$.

## B1. The criteria and the best threshold

Implement `gini(counts)` and `entropy(counts)` for an array whose **last axis** holds the class counts (any number of
leading axes: they are called on a whole matrix of left-child counts at once), and
`best_threshold(x, y, n_classes, criterion)` for one numeric feature `x` and integer labels `y`: sort `x`, take the
cumulative class counts (`np.cumsum` of the one-hot labels), evaluate the gain of **every** split between two *different*
consecutive values at once, and return `(threshold, gain)`, the threshold being the midpoint; `(None, 0.0)` if no split
has a positive gain.

In [ ]:
def gini(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum(axis=-1, keepdims=True)
    return 1.0 - np.sum(p**2, axis=-1)


def entropy(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum(axis=-1, keepdims=True)
    return -np.sum(np.where(p > 0, p * np.log2(np.where(p > 0, p, 1.0)), 0.0), axis=-1)


def best_threshold(x, y, n_classes, criterion=gini):
    order = np.argsort(x, kind="stable")
    xs, ys = x[order], y[order]
    n = len(ys)
    cum = np.cumsum(np.eye(n_classes)[ys], axis=0)           # class counts of the first i + 1 examples
    total = cum[-1]
    left, right = cum[:-1], total - cum[:-1]
    n_left = np.arange(1, n)
    gain = criterion(total) - (n_left / n) * criterion(left) - ((n - n_left) / n) * criterion(right)
    gain = np.where(xs[:-1] < xs[1:], gain, -np.inf)           # only between different values
    i = int(np.argmax(gain))
    if not np.isfinite(gain[i]) or gain[i] <= 1e-12:
        return None, 0.0
    return float((xs[i] + xs[i + 1]) / 2), float(gain[i])

**How we got there.** *Impurity:* the proportions are the counts divided by their sum over the last axis; for the Gini
index, one minus the sum of squares. For the entropy $0\log0$ must be $0$: `np.where(p > 0, p * log2(p), 0)` with a safe
argument inside the `log2` so that no warning appears. *The best threshold:* after sorting, the first $i+1$ examples go
left; the cumulative one-hot sum gives their class counts for **all** $i$ at once, and the right child is the total minus
the left; the gain is a vector over the $n-1$ boundaries. A boundary between two *equal* values is not a valid
question, so its gain is set to $-\infty$. Cost: one sort, then linear.

The driver reproduces the table of the lecture (the 14 days of "play tennis?") and compares your best thresholds on the
breast-cancer data with the root split of `DecisionTreeClassifier` (depth 1, same criterion).

In [ ]:
tennis = {"outlook": ["sunny", "sunny", "overcast", "rain", "rain", "rain", "overcast", "sunny", "sunny", "rain", "sunny", "overcast", "overcast", "rain"],
          "humidity": ["high", "high", "high", "high", "normal", "normal", "normal", "high", "normal", "normal", "normal", "high", "normal", "high"],
          "windy": ["no", "yes", "no", "no", "no", "yes", "yes", "no", "no", "no", "yes", "yes", "no", "yes"]}
play = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0])
parent = np.bincount(play, minlength=2)
rows = []
for feat, vals in tennis.items():
    vals = np.array(vals)
    child = sum((vals == v).mean() * entropy(np.bincount(play[vals == v], minlength=2)) for v in np.unique(vals))
    child_g = sum((vals == v).mean() * gini(np.bincount(play[vals == v], minlength=2)) for v in np.unique(vals))
    rows.append((feat, float(entropy(parent) - child), float(gini(parent) - child_g)))
print(f"parent (9 yes, 5 no): entropy {entropy(parent):.3f}, Gini {gini(parent):.3f}")
print(pl.DataFrame(rows, schema=["question", "gain (entropy)", "gain (Gini)"], orient="row"))

rows = []
for name, (X, y, names, tr) in {"breast cancer": (Xb, yb, names_b, trb), "german credit": (Xg, yg, names_g, trg)}.items():
    for crit, fn in (("gini", gini), ("entropy", entropy)):
        gains = [best_threshold(X[tr, j], y[tr], 2, fn) for j in range(X.shape[1])]
        j = int(np.argmax([g for _, g in gains]))
        ref = DecisionTreeClassifier(criterion=crit, max_depth=1, random_state=0).fit(X[tr], y[tr])
        rows.append((name, crit, names[j], gains[j][0], names[ref.tree_.feature[0]], float(ref.tree_.threshold[0])))
print(pl.DataFrame(rows, schema=["data", "criterion", "our feature", "our thr.", "sklearn feature", "sklearn thr."], orient="row"))

**Answers (B).**

1. Gains of the 14-day table: outlook 0.247 (entropy) and 0.116 (Gini), humidity 0.152 and 0.092, windy 0.048 and 0.031. Outlook is the root under both criteria. The parent has entropy 0.940 (9 yes, 5 no) and Gini 0.459.
2. On breast cancer the best question is `worst_concave_points <= 0.147` for both criteria; on German credit `checking_status_no checking <= 0.5` (the applicant has no checking account). Our feature and threshold equal scikit-learn's: the algorithm is the same.
3. Cost: one sort ($O(n\log n)$) and then a linear scan with cumulative counts; trying every feature at every node costs $O(d\,n\log n)$ for the first level and about the same for each of the next levels (the examples are partitioned). Without the cumulative sums, each of the $n-1$ thresholds would need an $O(n)$ count: $O(n^2)$.

# Part C — A tree from scratch

## C1. Grow it and use it

Implement `fit_tree(X, y, n_classes, max_depth, min_leaf, criterion, depth=0)` returning a dictionary node (see
`lab06_utils`): `counts`, `pred` (the majority class), `n`; and, for an internal node, `feature`, `threshold`, `left`,
`right`. Make a **leaf** when the node is pure, the depth limit is reached, or it has fewer than `2 * min_leaf` examples;
otherwise take the best (feature, threshold) over all features with `best_threshold`, **reject** splits that leave fewer
than `min_leaf` examples on a side, and recurse on the two parts. Implement `predict_tree(node, X)`: follow the questions
from the root.

In [ ]:
def fit_tree(X, y, n_classes, max_depth=5, min_leaf=1, criterion=gini, depth=0):
    counts = np.bincount(y, minlength=n_classes)
    node = {"counts": counts, "pred": int(counts.argmax()), "n": len(y)}
    if depth >= max_depth or counts.max() == len(y) or len(y) < 2 * min_leaf:
        return node
    best = (None, None, 0.0)
    for j in range(X.shape[1]):
        t, g = best_threshold(X[:, j], y, n_classes, criterion)
        if t is not None and g > best[2]:
            m = X[:, j] <= t
            if m.sum() >= min_leaf and (~m).sum() >= min_leaf:
                best = (j, t, g)
    if best[0] is None:
        return node
    j, t, g = best
    m = X[:, j] <= t
    node.update(feature=j, threshold=t, left=fit_tree(X[m], y[m], n_classes, max_depth, min_leaf, criterion, depth + 1),
                right=fit_tree(X[~m], y[~m], n_classes, max_depth, min_leaf, criterion, depth + 1))
    return node


def predict_tree(node, X):
    out = np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        n = node
        while "feature" in n:
            n = n["left"] if x[n["feature"]] <= n["threshold"] else n["right"]
        out[i] = n["pred"]
    return out

**How we got there.** The recursion has three parts. *Base case:* a leaf stores the class counts and the majority class
(the prediction). *Search:* for each feature the best threshold comes from Part B; we keep the feature with the largest
gain, but skip a split whose side would have fewer than `min_leaf` examples (the smallest allowed leaf). *Recurse:* the
examples are partitioned by the mask `x_j <= t` and each part grows its own subtree one level deeper. Prediction walks
one path from the root: at most `max_depth` comparisons per example, independent of the size of the training set.

In [ ]:
rows = []
for name, (X, y, tr, te, names) in {"breast cancer": (Xb, yb, trb, teb, names_b), "german credit": (Xg, yg, trg, teg, names_g)}.items():
    for depth in (2, 4, 6):
        t0 = time.time()
        ours = fit_tree(X[tr], y[tr], 2, max_depth=depth, min_leaf=1)
        t_ours = time.time() - t0
        ref = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X[tr], y[tr])
        pred_ours, pred_ref = predict_tree(ours, X[te]), ref.predict(X[te])
        rows.append((name, depth, float(np.mean(pred_ours == y[te])), float(ref.score(X[te], y[te])), float(np.mean(pred_ours == pred_ref)),
                     leaves_of(ours), int(ref.get_n_leaves()), t_ours))
print(pl.DataFrame(rows, schema=["data", "depth", "acc ours", "acc sklearn", "same pred.", "leaves ours", "leaves sk", "fit (s)"], orient="row"))

tree3 = fit_tree(Xb[trb], yb[trb], 2, max_depth=3)
print("\nOur depth-3 tree for breast cancer, as rules:")
print_tree(tree3, names_b, class_names=["malignant", "benign"])

**Answers (C).**

1. Our tree and `DecisionTreeClassifier` have the **same number of leaves** at every depth and agree on 97.9-100% of the test predictions (German credit depth 2 and 4: 100%). The few differences come from **ties** (two questions with exactly the same gain, resolved in another order) and from scikit-learn storing the features as `float32`, which moves a threshold by a rounding error.
2. The root question of the depth-3 tree is `worst_concave_points <= 0.147`; its left branch holds 294 of the 426 training tumours and is almost pure benign (244 of the 249 examples of its largest leaf). Two leaves contain **a single example** (`n = 1`): a rule built on one patient, an overfitting symptom that `min_leaf` would forbid.
3. Test accuracy for depths 2, 4 and 6: 0.930, 0.944, 0.937 (breast cancer) and 0.724, 0.724, 0.692 (German credit): more depth does not help, and it hurts on the noisier problem.

# Part D — Overfitting and the choice of the depth

## D1. Depth on validation data

Implement `best_depth(Xtr, ytr, Xva, yva, n_classes, depths, min_leaf)`: for each depth fit your tree on the training part
and measure the accuracy on the training and validation parts; return the depth with the best **validation** accuracy (the
smallest one on a tie) and the two curves as arrays.

In [ ]:
def best_depth(Xtr, ytr, Xva, yva, n_classes, depths, min_leaf=1):
    tr_acc, va_acc = [], []
    for d in depths:
        node = fit_tree(Xtr, ytr, n_classes, max_depth=d, min_leaf=min_leaf)
        tr_acc.append(np.mean(predict_tree(node, Xtr) == ytr))
        va_acc.append(np.mean(predict_tree(node, Xva) == yva))
    return depths[int(np.argmax(va_acc))], np.array(tr_acc), np.array(va_acc)

**How we got there.** Fit one tree per depth; the training accuracy can only grow with the depth (more questions fit the
training set better), the validation accuracy rises and then falls: the best depth is the maximum of the second curve.
`np.argmax` returns the first maximum, i.e. the smallest depth on a tie, which is also the simplest tree.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
depths = list(range(1, 13))
for ax, (name, X, y, tr, te) in zip(axes, (("breast cancer", Xb, yb, trb, teb), ("german credit", Xg, yg, trg, teg))):
    # the validation part is the second half of the training part; the test part is not touched
    h = len(tr) // 2
    d, tr_acc, va_acc = best_depth(X[tr[:h]], y[tr[:h]], X[tr[h:]], y[tr[h:]], 2, depths)
    ax.plot(depths, tr_acc, label="train")
    ax.plot(depths, va_acc, label="validation")
    ax.axvline(d, color="k", ls=":")
    ax.set(xlabel="max_depth", title=f"{name}: best depth {d}")
    final = fit_tree(X[tr], y[tr], 2, max_depth=d)
    print(f"{name}: best depth {d}; test accuracy of a tree of that depth (fitted on all the training part): {np.mean(predict_tree(final, X[te]) == y[te]):.3f}; "
          f"full tree {np.mean(predict_tree(fit_tree(X[tr], y[tr], 2, max_depth=50), X[te]) == y[te]):.3f}")
axes[0].legend()
plt.tight_layout()
plt.show()

**Cost-complexity pruning** (scikit-learn, given): the whole path of nested trees from one full tree, chosen by 5-fold
cross-validation on the training part.

In [ ]:
rows = []
for name, X, y, tr, te in (("breast cancer", Xb, yb, trb, teb), ("german credit", Xg, yg, trg, teg)):
    full = DecisionTreeClassifier(random_state=0).fit(X[tr], y[tr])
    alphas = np.unique(np.round(full.cost_complexity_pruning_path(X[tr], y[tr]).ccp_alphas, 6))
    grid = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas[:: max(1, len(alphas) // 30)]}, cv=5).fit(X[tr], y[tr])
    rows.append((name, int(full.get_n_leaves()), float(full.score(X[te], y[te])), int(grid.best_estimator_.get_n_leaves()),
                 float(grid.best_params_["ccp_alpha"]), float(grid.score(X[te], y[te]))))
print(pl.DataFrame(rows, schema=["data", "leaves, full", "test acc, full", "leaves, pruned", "best alpha", "test acc, pruned"], orient="row"))

**Answers (D).**

1. The best depth on the validation half is **2** for breast cancer and **1** for German credit. The full tree reaches a training accuracy of 1.000 and a test accuracy of 0.937 (breast cancer) and **0.628** (German credit, below the 0.70 of the majority answer!): the German-credit tree memorizes noise. The unlimited tree has 22 and 152 leaves.
2. Cost-complexity pruning chosen by cross-validation keeps **7** leaves (test 0.944, against 0.923 unpruned) on breast cancer and only **3** leaves on German credit (test 0.724, against 0.648). On German credit the best tree is almost a stump, and its 0.724 is only 2.4 points above the majority baseline: the attributes carry little signal, and a more complex model cannot create it.

# Part E — A regression tree

## E1. Variance reduction

For a numeric target a leaf predicts the **mean** of its examples and a split is judged by the reduction of the **sum of
squared errors** $\text{SSE} = \sum (y_i - \bar y)^2$. Implement `best_threshold_reg(x, y)` (the same sorted scan, with the
cumulative sums of $y$ and $y^2$: $\text{SSE} = \sum y^2 - (\sum y)^2/n$ for each side) returning `(threshold, gain)`, and
`fit_reg_tree(X, y, max_depth, min_leaf)` returning nodes with `value` (the mean), `pred` (the same), `n`, and
`feature`, `threshold`, `left`, `right` for internal ones; `predict_reg_tree` is given.

In [ ]:
def best_threshold_reg(x, y):
    order = np.argsort(x, kind="stable")
    xs, ys = x[order], y[order]
    n = len(ys)
    c1, c2 = np.cumsum(ys), np.cumsum(ys**2)
    n_left = np.arange(1, n)
    sse_left = c2[:-1] - c1[:-1] ** 2 / n_left
    sse_right = (c2[-1] - c2[:-1]) - (c1[-1] - c1[:-1]) ** 2 / (n - n_left)
    gain = (c2[-1] - c1[-1] ** 2 / n) - sse_left - sse_right
    gain = np.where(xs[:-1] < xs[1:], gain, -np.inf)
    i = int(np.argmax(gain))
    if not np.isfinite(gain[i]) or gain[i] <= 1e-12:
        return None, 0.0
    return float((xs[i] + xs[i + 1]) / 2), float(gain[i])


def fit_reg_tree(X, y, max_depth=4, min_leaf=5, depth=0):
    node = {"value": float(y.mean()), "pred": float(y.mean()), "n": len(y)}
    if depth >= max_depth or len(y) < 2 * min_leaf:
        return node
    best = (None, None, 0.0)
    for j in range(X.shape[1]):
        t, g = best_threshold_reg(X[:, j], y)
        if t is not None and g > best[2]:
            m = X[:, j] <= t
            if m.sum() >= min_leaf and (~m).sum() >= min_leaf:
                best = (j, t, g)
    if best[0] is None:
        return node
    j, t, g = best
    m = X[:, j] <= t
    node.update(feature=j, threshold=t, left=fit_reg_tree(X[m], y[m], max_depth, min_leaf, depth + 1),
                right=fit_reg_tree(X[~m], y[~m], max_depth, min_leaf, depth + 1))
    return node


def predict_reg_tree(node, X):
    out = np.empty(len(X))
    for i, x in enumerate(X):
        n = node
        while "feature" in n:
            n = n["left"] if x[n["feature"]] <= n["threshold"] else n["right"]
        out[i] = n["pred"]
    return out

**How we got there.** The SSE of a group is $\sum y^2 - (\sum y)^2/n$, so with the cumulative sums of $y$ and $y^2$ the
SSE of the left part of every split and (from the totals) of the right part are vectors; the gain is the SSE of the
parent minus the two. The tree is the classification tree with the leaf value changed from a majority to a mean and
the impurity changed from Gini to the SSE: *the algorithm is the same.*

In [ ]:
Xtr_c, Xte_c, ytr_c, yte_c = Xc[trc], Xc[tec], yc[trc], yc[tec]
rows, preds = [], {}
for depth in (2, 4, 6, 10):
    ours = fit_reg_tree(Xtr_c, ytr_c, max_depth=depth, min_leaf=5)
    ref = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=5, random_state=0).fit(Xtr_c, ytr_c)
    p_o, p_r = predict_reg_tree(ours, Xte_c), ref.predict(Xte_c)
    rows.append((depth, float(np.sqrt(np.mean((p_o - yte_c) ** 2))), float(np.sqrt(np.mean((p_r - yte_c) ** 2))), float(np.max(np.abs(p_o - p_r))), leaves_of(ours), int(ref.get_n_leaves())))
    if depth == 6:
        preds = {"ours, depth 6": p_o, "sklearn, depth 6": p_r}
lin = LinearRegression().fit(Xtr_c, ytr_c)
print(pl.DataFrame(rows, schema=["depth", "RMSE ours", "RMSE sklearn", "max |diff|", "leaves ours", "leaves sk"], orient="row"))
print(f"linear regression: RMSE {np.sqrt(np.mean((lin.predict(Xte_c) - yte_c) ** 2)):.2f} MPa")
plot_pred_vs_actual(yte_c, preds)

**Answers (E).**

1. At depths 2 and 4 our regression tree equals scikit-learn's (difference 0.000); at depths 6 and 10 the trees differ in 48 against 52 and 105 against 119 leaves (ties and `float32`), and the largest prediction difference grows to 5.9 and 23 MPa while the RMSE stays close (7.94 against 7.86 and 6.79 against 7.06).
2. The tree reaches **6.8 MPa** at depth 10, the linear regression **11.4 MPa** and the mean 15.8 MPa: the strength depends on the mixture in a non-linear way with interactions (cement with age, water with superplasticizer), which a tree captures with its questions. The predicted-versus-actual plot shows the staircase: a tree predicts only as many different values as it has leaves.

# Part F — k-Nearest Neighbours with `jax.numpy`

## F1. Distances, neighbours, vote

Implement `pairwise_sq_dists(A, B)` (shape `(len(A), len(B))`, with broadcasting: $\lVert a-b\rVert^2 = \lVert a\rVert^2 +
\lVert b\rVert^2 - 2a^\top b$) and `knn_predict(Xtr, ytr, Xte, k, n_classes, weights="uniform")`: for every test point
take the $k$ nearest training points (`jax.lax.top_k` of the *negative* distances), and let them vote (the sum of the
one-hot labels; with `weights="distance"` each vote is multiplied by $1/(d + 10^{-9})$). Return the winning class.

In [ ]:
def pairwise_sq_dists(A, B):
    return jnp.sum(A**2, axis=1)[:, None] + jnp.sum(B**2, axis=1)[None, :] - 2.0 * A @ B.T


def knn_predict(Xtr, ytr, Xte, k=5, n_classes=2, weights="uniform"):
    d2 = jnp.maximum(pairwise_sq_dists(jnp.asarray(Xte), jnp.asarray(Xtr)), 0.0)
    neg, idx = jax.lax.top_k(-d2, k)
    votes = jax.nn.one_hot(jnp.asarray(ytr)[idx], n_classes)
    if weights == "distance":
        votes = votes * (1.0 / (jnp.sqrt(-neg) + 1e-9))[:, :, None]
    return np.asarray(votes.sum(axis=1).argmax(axis=1))

**How we got there.** The expansion of the squared norm turns the all-pairs distances into one matrix product plus two
vectors of squared norms broadcast over rows and columns; rounding can make a tiny value negative, so it is clipped at
zero before the square root. `top_k` returns the $k$ *largest* entries: applied to $-d^2$ it returns the $k$ nearest, with
their indices. One-hot labels of the neighbours, summed over the $k$ axis, count the votes; the weighted vote multiplies
each neighbour by the inverse of its distance.

In [ ]:
mu_b, sd_b = fit_standardizer(Xb[trb])
A_b, B_b = standardize(Xb[trb], mu_b, sd_b), standardize(Xb[teb], mu_b, sd_b)
mu_g, sd_g = fit_standardizer(Xg[trg])
A_g, B_g = standardize(Xg[trg], mu_g, sd_g), standardize(Xg[teg], mu_g, sd_g)
D_ours = np.asarray(pairwise_sq_dists(jnp.asarray(A_b[:20]), jnp.asarray(A_b[20:60])))
D_ref = ((A_b[:20, None, :] - A_b[None, 20:60, :]) ** 2).sum(-1)
compare("pairwise squared distances", D_ours, D_ref)
rows = []
for name, (A, B, ya, yb_) in {"breast cancer": (A_b, B_b, yb[trb], yb[teb]), "german credit": (A_g, B_g, yg[trg], yg[teg])}.items():
    for k in (1, 5, 15):
        for w in ("uniform", "distance"):
            ours = knn_predict(A, ya, B, k, 2, w)
            ref = KNeighborsClassifier(k, weights=w).fit(A, ya)
            rows.append((name, k, w, float(np.mean(ours == yb_)), float(ref.score(B, yb_)), float(np.mean(ours == ref.predict(B)))))
print(pl.DataFrame(rows, schema=["data", "k", "weights", "acc ours", "acc sklearn", "same pred."], orient="row"))

**Scaling, $k$ and irrelevant features** (all with your `knn_predict`): the accuracy with raw and standardized features,
the curve of $k$ on the moons, and the accuracy when $r$ columns of noise are added to the breast-cancer data.

In [ ]:
raw = np.mean(knn_predict(Xb[trb], yb[trb], Xb[teb], 5, 2) == yb[teb])
std = np.mean(knn_predict(A_b, yb[trb], B_b, 5, 2) == yb[teb])
print(f"breast cancer, k = 5: raw features {raw:.3f}, standardized {std:.3f}")

Xm, ym = make_moons(300, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(3000, noise=0.3, random_state=1)
ks = [1, 3, 5, 9, 15, 25, 51, 101, 201]
tr_acc = [np.mean(knn_predict(Xm, ym, Xm, k, 2) == ym) for k in ks]
te_acc = [np.mean(knn_predict(Xm, ym, Xm_te, k, 2) == ym_te) for k in ks]
plot_curves(ks, {"train": np.array(tr_acc), "test": np.array(te_acc)}, "k", logx=True)

rng = np.random.default_rng(0)
rows = []
for r in (0, 10, 50, 200, 1000):
    A = np.c_[A_b, rng.normal(size=(len(A_b), r))]
    B = np.c_[B_b, rng.normal(size=(len(B_b), r))]
    knn_acc = np.mean(knn_predict(A, yb[trb], B, 5, 2) == yb[teb])
    tree_acc = np.mean(predict_tree(fit_tree(A, yb[trb], 2, max_depth=4, min_leaf=5), B) == yb[teb])
    lr = LogisticRegression(max_iter=3000).fit(A, yb[trb]).score(B, yb[teb])
    rows.append((r, float(knn_acc), float(tree_acc), float(lr)))
print(pl.DataFrame(rows, schema=["noise features", "KNN k=5", "our tree", "logistic"], orient="row"))

**Answers (F).**

1. Our squared distances equal the direct computation to $10^{-13}$, and our KNN makes **the same predictions as scikit-learn in 100%** of the cases for every $k$ and both weightings (the algorithm has no randomness: the same neighbours, the same vote).
2. Scaling: on breast cancer the accuracy goes from **0.958 to 0.972** with standardization (the area, thousands of units, decides the raw distance). The moons curve shows the trade-off: $k = 1$ has training accuracy 1.0 and the lowest test accuracy; the test accuracy rises to a plateau around $k = 15$-51 and falls when $k$ approaches $n$ (the vote tends to the majority class).
3. With 1000 noise columns the KNN falls from 0.972 to **0.817**, the tree from 0.958 to 0.923 and logistic regression from 0.986 to 0.915: the distance gives the same weight to a useful and to a useless feature, while a tree can ignore a column that never gives a gain and a linear model can give it a small weight. KNN has no way to *select* features.

# Part G — KNN for regression

## G1. The local mean

Implement `knn_regress(Xtr, ytr, Xte, k)`: the mean target of the $k$ nearest training points (reuse
`pairwise_sq_dists` and `jax.lax.top_k`).

In [ ]:
def knn_regress(Xtr, ytr, Xte, k=5):
    d2 = jnp.maximum(pairwise_sq_dists(jnp.asarray(Xte), jnp.asarray(Xtr)), 0.0)
    _, idx = jax.lax.top_k(-d2, k)
    return np.asarray(jnp.asarray(ytr)[idx].mean(axis=1))

**How we got there.** The neighbours are found exactly as for classification; instead of a vote the prediction is the mean
of their targets. With $k = 1$ it interpolates the data (zero training error); a large $k$ averages away the noise and
flattens the function.

In [ ]:
mu_c, sd_c = fit_standardizer(Xtr_c)
A_c, B_c = standardize(Xtr_c, mu_c, sd_c), standardize(Xte_c, mu_c, sd_c)
rows = []
for k in (1, 3, 5, 10, 20, 50):
    p = knn_regress(A_c, ytr_c, B_c, k)
    ref = KNeighborsRegressor(k).fit(A_c, ytr_c).predict(B_c)
    rows.append((k, float(np.sqrt(np.mean((p - yte_c) ** 2))), float(np.sqrt(np.mean((ref - yte_c) ** 2))), float(np.max(np.abs(p - ref)))))
print(pl.DataFrame(rows, schema=["k", "RMSE ours", "RMSE sklearn", "max |diff|"], orient="row"))
print(f"concrete, test RMSE: linear {np.sqrt(np.mean((lin.predict(Xte_c) - yte_c) ** 2)):.2f}, "
      f"tree (depth 10) {np.sqrt(np.mean((predict_reg_tree(fit_reg_tree(Xtr_c, ytr_c, 10, 5), Xte_c) - yte_c) ** 2)):.2f}, "
      f"KNN k=5 {np.sqrt(np.mean((knn_regress(A_c, ytr_c, B_c, 5) - yte_c) ** 2)):.2f} MPa")

**Answers (G).** Our local mean agrees with `KNeighborsRegressor` (the RMSEs match to two decimals; the maximum difference of one prediction comes from equal distances broken in another order). The RMSE is 9.1 MPa for $k = 1$, **8.6 for $k = 3$-5**, 9.3 for $k = 10$ and 10.4 for $k = 50$. On concrete the ranking is tree (6.8) < KNN (8.6) < linear (11.4): the tree finds the interactions; KNN is local but suffers from eight unscaled-importance features (all weigh the same); the line cannot bend.

# Part H — Challenge: the symbolist and the analogizer (home)

Choose a data set of Project 1. Compare a **decision tree** (with `min_samples_leaf` and `ccp_alpha` chosen by
cross-validation) and a **KNN** (the number of neighbours chosen by cross-validation, standardization inside the pipeline)
with logistic regression on repeated stratified splits, and decide with the corrected paired $t$-test of Class 05
whether any difference is real.

## H1. Repeated splits

Implement `repeated_scores(make_model, X, y, n_rep, test_frac)`: for `rep in range(n_rep)` split with
`stratified_split(y, test_frac, np.random.default_rng(rep))`, standardize with the **training** statistics, fit
`make_model()` and return the array of test accuracies.

In [ ]:
def repeated_scores(make_model, X, y, n_rep=10, test_frac=0.25):
    scores = []
    for rep in range(n_rep):
        tr, te = stratified_split(y, test_frac, np.random.default_rng(rep))
        mu, sd = fit_standardizer(X[tr])
        model = make_model().fit(standardize(X[tr], mu, sd), y[tr])
        scores.append(float(model.score(standardize(X[te], mu, sd), y[te])))
    return np.array(scores)

**How we got there.** Each repetition is a complete experiment: a new stratified split, statistics from the training part
only (no leakage), a fresh model. The accuracies are *paired* across learners because the seed fixes the split.

In [ ]:
results = {
    "logistic": repeated_scores(lambda: LogisticRegression(max_iter=3000), Xg, yg),
    "tree (tuned)": repeated_scores(lambda: GridSearchCV(DecisionTreeClassifier(random_state=0), {"min_samples_leaf": [1, 5, 10, 20, 50]}, cv=5), Xg, yg),
    "KNN (tuned)": repeated_scores(lambda: GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [1, 3, 5, 9, 15, 25]}, cv=5), Xg, yg),
}
print(pl.DataFrame([(k, float(v.mean()), float(v.std(ddof=1))) for k, v in results.items()], schema=["German credit", "mean accuracy", "sd"], orient="row"))
for a, b in (("KNN (tuned)", "tree (tuned)"), ("KNN (tuned)", "logistic"), ("tree (tuned)", "logistic")):
    d = results[a] - results[b]
    t = d.mean() / np.sqrt((1 / len(d) + 1 / 3) * d.var(ddof=1))
    print(f"{a} - {b}: mean difference {d.mean():+.4f}, t = {t:.2f}, p = {2 * stats.t.sf(abs(t), len(d) - 1):.3f}")

**Answers (H).** On German credit (10 splits): logistic regression 0.750 (sd 0.018), tuned KNN 0.732 (0.024), tuned tree 0.704 (0.031). The corrected test finds **no significant difference** between KNN and the tree ($+0.028$, $p = 0.35$) or KNN and logistic regression ($-0.018$, $p = 0.22$), and a borderline one between the tree and logistic regression ($-0.046$, $p = 0.059$). The conclusion is *modest*: on this small, noisy data set the simplest model is not worse, the tree has the largest spread (a high-variance learner), and none of the three is clearly better than the 0.70 baseline plus a few points.